In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

In [0]:
bronze_root = "abfss://bronze@grtadlsdev.dfs.core.windows.net"

# find latest folder by time modified
folders = dbutils.fs.ls(bronze_root)
latest_folder = sorted(folders, key=lambda f: f.modificationTime, reverse=True)[0].name
bronze_path = f"{bronze_root}/{latest_folder}"
print("Using bronze path:", bronze_path)

In [0]:
for f in dbutils.fs.ls(bronze_path):
    print(repr(f.name), f.path)

In [0]:
# strip extension
files = {}
for f in dbutils.fs.ls(bronze_path):
    name = f.name.rstrip('/')                # remove trailing slash if present
    if name.lower().endswith(".txt"):        # case-insensitive match
        key = name.rsplit(".", 1)[0]
        files[key] = f.path.rstrip('/')     # normalize path (no trailing slash)

print("bronze_path:", bronze_path)
print("Discovered tables:", list(files.keys()))

In [0]:
# create bronze schema if it doesn't exist
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

# loop over files and write to delta tables
for table_name, path in files.items():
    (
        spark.read.format("csv")
            .option("header", "true")
            .option("inferSchema", "true")
            .load(path)
            .write.format("delta")
            .mode("overwrite")
            .saveAsTable(f"bronze.{table_name}")
    )

In [0]:
# validate the tables
spark.sql("SHOW TABLES IN bronze").show(truncate=False)

# row count
for t in files.keys():
    count = spark.table(f"bronze.{t}").count()
    print(f"{t}: {count} rows")

# peek at the schema
for t in files.keys():
    print(f"\nSchema for bronze.{t}")
    spark.table(f"bronze.{t}").printSchema()